# Job 2 — LLM parser: Qwen3-8B-Base + LoRA (English text → UMR)
Same data splits (A, B) and the same metrics as Job 1, so every number compares directly with your AMRBART results and the paper (SETUP, LREC 2026, Table 5).

| Part | What it does | Time (A100 80GB) |
|---|---|---|
| 0 Setup | GPU, Drive, installs, code files | ~5 min |
| 1 Data | load Job 1's saved splits, one-line graph format, length check | ~3 min |
| 2 Train A | LoRA fine-tune on split A, test, compare with AMRBART + paper | ~1.5–2 h |
| 3 Train B | same on clean split B | ~1–1.5 h |
| 4 Compare | final table + where the LLM wins/loses vs AMRBART on non-Minecraft | ~5 min |

**Needs from Job 1 (already on Drive):** `umr_job1/data/A`, `umr_job1/data/B`, and AMRBART results in `umr_job1/runs/amrbart_A|B/`.

**Decisions** (🔧 = not from the paper; the paper has no LLM parser, so most choices here are ours):
model Qwen3-8B-Base (you chose); one-line PENMAN with variables renamed z0, z1… (you chose); splits A and B (you chose); no constrained decoding, parentheses repaired after generation (you chose); LoRA r=16/α=32 on all linear layers, lr 2e-4, 3 epochs, batch 16, cosine schedule, greedy decoding (common LoRA defaults).

# Part 0 — Setup

### 0.1 GPU

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-80GB, 81920 MiB


### 0.2 Drive (same folder as Job 1)

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/umr_job1'
WORK  = '/content/job2'
os.makedirs(WORK, exist_ok=True)
%cd {WORK}
print(os.listdir(f'{DRIVE}/data'), os.listdir(f'{DRIVE}/runs'))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/job2
['A', 'B'] ['amrbart_A', 'amrbart_B']


### 0.3 Install
Recent `transformers` (Qwen3 needs ≥ 4.51) — this is a **fresh runtime**, so it doesn't clash with Job 1's pinned 4.46.3. AMRBART is not needed here.

In [3]:
!pip -q install -U "transformers>=4.51" peft accelerate penman smatch mip
!pip -q install git+https://github.com/flipz357/smatchpp.git
!git clone -q https://github.com/umr4nlp/ancast.git && pip -q install -e ancast
import sys; sys.path.insert(0, WORK); sys.path.insert(0, f'{WORK}/ancast')
import warnings, logging; warnings.filterwarnings('ignore'); logging.getLogger('penman').setLevel(logging.ERROR)
# !pip -q uninstall -y torchao

  Preparing metadata (setup.py) ... done
fatal: destination path 'ancast' already exists and is not an empty directory.


### 0.4 Code files
- `umr_tools.py` — **identical to Job 1** (data helpers + metrics), so scoring is the same.
- `llm_tools.py` — new: graph format, LoRA training loop, generation.

In [4]:
%%writefile umr_tools.py
"""
umr_tools.py - data, split and evaluation helpers for replicating SETUP
(Markle, Gutierrez Bach & Wein, LREC 2026) with AMRBART.

Sections of the paper referenced below:
  Sec 3.3 Data, Sec 3.4 Data split, Sec 3.5 Metrics, Sec 4.1 Minecraft breakdown.
"""
import glob
import json
import os
import random
import re
import subprocess
import sys
from collections import Counter, defaultdict

import logging
import penman
logging.getLogger("penman").setLevel(logging.ERROR)

BLOCK_SEP = re.compile(r"^#{20,}\s*$", re.M)
MINECRAFT_RE = re.compile(r"\[Builder|<Builder>|<Architect>|\[Architect")
BACKOFF_GRAPH = "(z0 / empty-graph)"   # used when a prediction cannot be parsed


# ---------------------------------------------------------------- reading
def _section(block, start, ends):
    """Text between a '# start' header and the next '# end' header."""
    i = block.find(start)
    if i < 0:
        return ""
    txt = block[i + len(start):]
    for e in ends:
        j = txt.find(e)
        if j >= 0:
            txt = txt[:j]
    return txt.strip()


def read_umr_file(path):
    """Yield one record per sentence block of a .umr file."""
    raw = open(path, encoding="utf-8").read()
    doc = os.path.basename(path)
    for k, block in enumerate(b for b in BLOCK_SEP.split(raw) if b.strip()):
        m = re.search(r"^Words:[ \t]*(.*)$", block, re.M)
        if m:
            sent = " ".join(m.group(1).split())
        else:  # some files only have '# :: snt1  text'
            m = re.search(r"^#[ \t]*::[ \t]*snt\d+[ \t]*(.*)$", block, re.M)
            sent = " ".join(m.group(1).split()) if m else ""
        graph = _section(block, "# sentence level graph:",
                         ["# alignment:", "# document level annotation:"])
        meta = re.search(r"sent_id\s*=\s*(\S+)", block)
        yield {"doc": doc, "idx": k, "sent_id": meta.group(1) if meta else f"{doc}#{k}",
               "sentence": sent, "graph_raw": graph}


def balance_parens(s):
    """Fix the most common annotation error: too many / too few ')' (parens inside quotes ignored)."""
    depth, out, in_q = 0, [], False
    for ch in s:
        if ch == '"':
            in_q = not in_q
        elif not in_q and ch == "(":
            depth += 1
        elif not in_q and ch == ")":
            if depth == 0:          # extra closing paren -> drop it
                continue
            depth -= 1
        out.append(ch)
    return "".join(out).rstrip() + ")" * depth


def clean_graph(s):
    """Return a canonical PENMAN string, or None if the graph is empty/unparseable."""
    s = (s or "").strip()
    if not s or s.replace(" ", "") in ("()", "(())"):
        return None
    try:
        g = penman.decode(balance_parens(s))
        if len(g.instances()) == 0:
            return None
        g.metadata = {}
        return penman.encode(g, indent=4)
    except Exception:
        return None


def is_minecraft(sentence):
    """Sec 4.1: Minecraft = sentence contains a Builder/Architect tag."""
    return bool(MINECRAFT_RE.search(sentence))


def load_english(umr_dir):
    """Read every English .umr file, clean graphs, return (records, stats)."""
    files = sorted(f for f in glob.glob(os.path.join(umr_dir, "**", "*.umr"), recursive=True)
                   if "english" in f.lower())
    if not files:
        raise FileNotFoundError(f"no English .umr files under {umr_dir}")
    recs, stats = [], Counter()
    for f in files:
        for r in read_umr_file(f):
            stats["blocks"] += 1
            if not r["sentence"]:
                stats["no_sentence"] += 1
                continue
            raw_ok = True
            try:
                penman.decode(r["graph_raw"])
            except Exception:
                raw_ok = False
            g = clean_graph(r["graph_raw"])
            if g is None:
                stats["empty_or_broken_graph"] += 1
                continue
            if not raw_ok:
                stats["fixed_by_paren_balancing"] += 1
            r["umr"] = g
            r["is_minecraft"] = is_minecraft(r["sentence"])
            del r["graph_raw"]
            recs.append(r)
    stats["files"] = len(files)
    stats["kept"] = len(recs)
    stats["minecraft"] = sum(r["is_minecraft"] for r in recs)
    return recs, stats


# ---------------------------------------------------------------- splits
def read_gold_sentences(path):
    """Sentences (# ::snt lines) of an AMR-style file, e.g. the authors' dev gold file."""
    return [" ".join(l[len("# ::snt"):].split())
            for l in open(path, encoding="utf-8") if l.startswith("# ::snt")]


def split_paper_style(recs, author_dev_sents, test_size=2941, seed=42):
    """Split A (Sec 3.4 replication): the authors' exact dev set + random sentence-level test."""
    need = Counter(author_dev_sents)
    dev, rest = [], []
    for r in recs:
        if need[r["sentence"]] > 0:
            need[r["sentence"]] -= 1
            dev.append(r)
        else:
            rest.append(r)
    rng = random.Random(seed)
    rng.shuffle(rest)
    test, train = rest[:test_size], rest[test_size:]
    unmatched = sum(need.values())
    return {"train": train, "dev": dev, "test": test}, unmatched


def split_clean(recs, dev_frac=0.1, test_frac=0.1, seed=42):
    """Split B (ours): split by DOCUMENT, then drop dev/test sentences whose exact
    text also occurs in train (removes the repeated Minecraft lines)."""
    by_doc = defaultdict(list)
    for r in recs:
        by_doc[r["doc"]].append(r)
    docs = sorted(by_doc)
    random.Random(seed).shuffle(docs)
    n = len(recs)
    parts, acc, cur = {"test": [], "dev": [], "train": []}, 0, "test"
    for d in docs:
        if cur == "test" and acc >= test_frac * n:
            cur, acc = "dev", 0
        if cur == "dev" and acc >= dev_frac * n:
            cur = "train"
        parts[cur].extend(by_doc[d])
        acc += len(by_doc[d])
    train_sents = {r["sentence"] for r in parts["train"]}
    removed = {}
    for k in ("dev", "test"):
        before = len(parts[k])
        parts[k] = [r for r in parts[k] if r["sentence"] not in train_sents]
        removed[k] = before - len(parts[k])
    return parts, removed


def mask_numbers(s):
    return re.sub(r"-?\d+", "N", s)


def leakage_report(split):
    """How many test sentences already appear in train (exactly / up to numbers)."""
    tr = {r["sentence"] for r in split["train"]}
    trm = {mask_numbers(s) for s in tr}
    out = {}
    for k in ("dev", "test"):
        xs = split[k]
        if not xs:
            continue
        ex = sum(r["sentence"] in tr for r in xs)
        tm = sum(mask_numbers(r["sentence"]) in trm for r in xs)
        out[k] = {"n": len(xs), "exact_in_train_%": round(100 * ex / len(xs), 1),
                  "template_in_train_%": round(100 * tm / len(xs), 1)}
    return out


def save_jsonl(recs, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in recs:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]


# ---------------------------------------------------------------- linearization
def _tokenize_encoded_graph(encoded):
    lin = re.sub(r"(\".+?\")", r" \1 ", encoded)
    pieces = []
    for p in lin.split():
        if p.startswith('"') and p.endswith('"'):
            pieces.append(p)
        else:
            for a, b in (("(", " ( "), (")", " ) "), (":", " :"), ("/", " / ")):
                p = p.replace(a, b)
            pieces.append(p.strip())
    return re.sub(r"\s+", " ", " ".join(pieces)).strip().split(" ")


def linearize(penman_str):
    """DFS linearization with <pointer:i> variables - same scheme AMRBART was trained on
    (AMR-Process `dfs_linearize`), so the pretrained parser sees a familiar format."""
    g = penman.decode(penman_str)
    g.metadata = {}
    toks = _tokenize_encoded_graph(penman.encode(g))
    remap = {}
    for i in range(1, len(toks)):
        if toks[i] == "/" and toks[i - 1] not in remap:
            remap[toks[i - 1]] = f"<pointer:{len(remap)}>"
    out, i = [toks[0]], 1
    while i < len(toks):
        nxt, lst = toks[i], out[-1]
        if nxt in remap:
            if lst == "(" and i + 1 < len(toks) and toks[i + 1] == "/":
                nxt = remap[nxt]
                i += 1
            elif lst.startswith(":"):
                nxt = remap[nxt]
        out.append(nxt)
        i += 1
    return out


# ---------------------------------------------------------------- evaluation
def to_valid(graph_str):
    """Parseable graph or the backoff graph, so no sentence is silently skipped by a scorer.
    A graph counts as valid only if penman AND smatch's own parser both accept it and
    every node has a concept."""
    try:
        g = penman.decode(graph_str)
        # drop empty values like :op1 "" (they carry no meaning and crash AnCast)
        g = penman.Graph([t for t in g.triples if t[2] not in ('""', "")], top=g.top)
        if not g.instances() or any(i.target in (None, "") for i in g.instances()):
            raise ValueError("node without concept")
        g.metadata = {}
        s = penman.encode(g, indent=4)
        import io, contextlib, amr as smatch_amr          # smatch's parser
        with contextlib.redirect_stderr(io.StringIO()):
            if smatch_amr.AMR.parse_AMR_line(" ".join(s.split())) is None:
                raise ValueError("smatch cannot parse")
        return s, True
    except Exception:
        return BACKOFF_GRAPH, False


def write_amr_style(sents, graphs, path):
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, g) in enumerate(zip(sents, graphs)):
            f.write(f"# ::id {i}\n# ::snt {s}\n{g}\n\n")


def score_smatch(pred_path, gold_path):
    import smatch
    with open(pred_path, encoding="utf-8") as p, open(gold_path, encoding="utf-8") as g:
        P, R, F = next(smatch.score_amr_pairs(p, g))
    return 100 * F


def score_smatchpp(pred_path, gold_path):
    """SMATCH++ with optimal (ILP) alignment, generic standardization, micro F1."""
    cmd = [sys.executable, "-m", "smatchpp", "-a", pred_path, "-b", gold_path,
           "-solver", "ilp", "-graph_type", "generic",
           "-score_dimension", "main", "-score_type", "micro"]
    out = subprocess.run(cmd, capture_output=True, text=True).stdout
    m = re.search(r"main ---->\s+F1:\s*([\d.]+)", out)
    return float(m.group(1)) if m else float("nan"), out


def write_umr_style(sents, graphs, path):
    """UMR block format that AnCast expects with data_format='umr'."""
    with open(path, "w", encoding="utf-8") as f:
        for i, (s, g) in enumerate(zip(sents, graphs), 1):
            f.write(f"# :: snt{i}\t{s}\n\n# sentence level graph:\n{g}\n\n"
                    f"# alignment:\n\n# document level annotation:\n\n\n")


def score_ancast(pred_path, gold_path, data_format="umr"):
    """AnCast (sentence scope). data_format='umr' scores UMR attributes such as
    :aspect / :refer-number; data_format='amr' silently ignores them (checked by us)."""
    from ancast import evaluate
    import logging
    logging.getLogger("ancast").setLevel(logging.ERROR)
    # pass FILE PATHS: AnCast only splits files into sentences; a raw string is
    # treated as one block and only its first sentence gets scored.
    res = evaluate(pred_path, gold_path, scope="snt", data_format=data_format,
                   output_csv_as_string=True)
    f = res[0] if isinstance(res, tuple) else res
    while isinstance(f, (list, tuple)):
        f = f[0]
    return 100 * float(f)


def evaluate_all(sents, preds, golds, flags, out_dir, name="test", run_smatchpp=True):
    """Scores for all / Minecraft / non-Minecraft (Sec 3.5 metrics + Sec 4.1 breakdown)."""
    os.makedirs(out_dir, exist_ok=True)
    fixed = [to_valid(p) for p in preds]
    preds_v = [g for g, _ in fixed]
    n_invalid = sum(not ok for _, ok in fixed)
    rows = {}
    for subset, keep in (("all", lambda f: True), ("minecraft", lambda f: f),
                         ("non_minecraft", lambda f: not f)):
        idx = [i for i, f in enumerate(flags) if keep(f)]
        if not idx:
            continue
        pp = os.path.join(out_dir, f"{name}_{subset}.pred.txt")
        gp = os.path.join(out_dir, f"{name}_{subset}.gold.txt")
        write_amr_style([sents[i] for i in idx], [preds_v[i] for i in idx], pp)
        write_amr_style([sents[i] for i in idx], [golds[i] for i in idx], gp)
        up, ug = pp.replace(".txt", ".umrfmt"), gp.replace(".txt", ".umrfmt")
        write_umr_style([sents[i] for i in idx], [preds_v[i] for i in idx], up)
        write_umr_style([sents[i] for i in idx], [golds[i] for i in idx], ug)
        row = {"n": len(idx), "AnCast": score_ancast(up, ug, "umr"),
               "AnCast_amrfmt": score_ancast(pp, gp, "amr"), "SMATCH": score_smatch(pp, gp)}
        if run_smatchpp:
            row["SMATCH++"] = score_smatchpp(pp, gp)[0]
        rows[subset] = {k: round(v, 2) if isinstance(v, float) else v for k, v in row.items()}
    rows["invalid_predictions"] = n_invalid
    with open(os.path.join(out_dir, f"{name}_scores.json"), "w") as f:
        json.dump(rows, f, indent=2)
    return rows

Overwriting umr_tools.py


In [5]:
%%writefile llm_tools.py
"""
llm_tools.py - Job 2: fine-tune a decoder-only LLM (Qwen3-8B-Base) with LoRA as a
text -> UMR parser. Same data splits and metrics as Job 1 so numbers compare directly.
"""
import json
import math
import os
import random
import time

import penman
import torch
import torch.nn.functional as F

import umr_tools as U

MODEL_NAME = "Qwen/Qwen3-8B-Base"
PROMPT = "Parse the sentence into a UMR graph.\nSentence: {sent}\nUMR:"


# ---------------------------------------------------------------- graph format
def to_oneline(penman_str):
    """One-line PENMAN with variables renamed z0, z1, ... in order of appearance.
    Annotator variable names (s34a, s34s ...) carry sentence ids = noise for the model."""
    g = penman.decode(penman_str)
    order = []
    for src, _, tgt in g.triples:
        for v in (src, tgt):
            if v in g.variables() and v not in order:
                order.append(v)
    ren = {v: f"z{i}" for i, v in enumerate(order)}
    tri = [(ren.get(s, s), r, ren.get(t, t)) for s, r, t in g.triples]
    g2 = penman.Graph(tri, top=ren[g.top])
    return penman.encode(g2, indent=None, compact=True)


def from_output(text):
    """Model text -> (penman_string, ok). Keeps the first line, repairs parentheses."""
    line = text.strip().split("\n")[0].strip()
    return U.to_valid(U.balance_parens(line))


# ---------------------------------------------------------------- data
def build_features(recs, tok, max_len=1280):
    feats, too_long = [], 0
    for r in recs:
        p = tok(PROMPT.format(sent=r["sentence"]), add_special_tokens=False)["input_ids"]
        t = tok(" " + to_oneline(r["umr"]), add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        if len(p) + len(t) > max_len:
            too_long += 1
            continue
        # loss only on the graph tokens (prompt positions masked with -100)
        feats.append({"input_ids": p + t, "labels": [-100] * len(p) + t})
    return feats, too_long


def batches(feats, bs, shuffle, seed=0):
    idx = list(range(len(feats)))
    if shuffle:
        random.Random(seed).shuffle(idx)
    chunk = bs * 50
    out = []
    for c in range(0, len(idx), chunk):
        part = sorted(idx[c:c + chunk], key=lambda i: len(feats[i]["input_ids"]))
        out += [part[i:i + bs] for i in range(0, len(part), bs)]
    if shuffle:
        random.Random(seed + 1).shuffle(out)
    return out


def collate(items, pad_id, device):
    L = max(len(x["input_ids"]) for x in items)
    inp = torch.full((len(items), L), pad_id)
    lab = torch.full((len(items), L), -100)
    att = torch.zeros((len(items), L), dtype=torch.long)
    for i, x in enumerate(items):          # right padding for training
        n = len(x["input_ids"])
        inp[i, :n] = torch.tensor(x["input_ids"])
        lab[i, :n] = torch.tensor(x["labels"])
        att[i, :n] = 1
    return inp.to(device), att.to(device), lab.to(device)


# ---------------------------------------------------------------- model
def load_model(name=MODEL_NAME, lora_r=16, lora_alpha=32, lora_dropout=0.05, adapter=None):
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import LoraConfig, PeftModel, get_peft_model
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16,
                                                 attn_implementation="sdpa").to("cuda")
    if adapter:
        model = PeftModel.from_pretrained(model, adapter, is_trainable=True)
    else:
        cfg = LoraConfig(r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout, bias="none",
                         task_type="CAUSAL_LM",
                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                                         "gate_proj", "up_proj", "down_proj"])
        model = get_peft_model(model, cfg)
    model.print_trainable_parameters()
    return model, tok


# ---------------------------------------------------------------- generation
@torch.no_grad()
def predict(model, tok, sentences, bs=32, max_new_tokens=768, device="cuda"):
    """Greedy decoding. Returns list of (penman_string, ok)."""
    model.eval()
    prompts = [tok(PROMPT.format(sent=s), add_special_tokens=False)["input_ids"] for s in sentences]
    order = sorted(range(len(prompts)), key=lambda i: len(prompts[i]))
    out = [None] * len(prompts)
    nl = tok("\n", add_special_tokens=False)["input_ids"]
    stop = [tok.eos_token_id] + (nl if len(nl) == 1 else [])
    for b in range(0, len(order), bs):
        idx = order[b:b + bs]
        L = max(len(prompts[i]) for i in idx)
        inp = torch.full((len(idx), L), tok.pad_token_id)
        att = torch.zeros((len(idx), L), dtype=torch.long)
        for j, i in enumerate(idx):        # LEFT padding for generation
            n = len(prompts[i])
            inp[j, L - n:] = torch.tensor(prompts[i])
            att[j, L - n:] = 1
        gen = model.generate(input_ids=inp.to(device), attention_mask=att.to(device),
                             max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                             eos_token_id=stop, pad_token_id=tok.pad_token_id)
        for j, i in enumerate(idx):
            text = tok.decode(gen[j, L:], skip_special_tokens=True)
            out[i] = from_output(text)
    return out


def dev_smatch(model, tok, dev_sample, out_dir, device="cuda"):
    preds = predict(model, tok, [r["sentence"] for r in dev_sample], device=device)
    os.makedirs(out_dir, exist_ok=True)
    p, g = os.path.join(out_dir, "p.txt"), os.path.join(out_dir, "g.txt")
    U.write_amr_style([r["sentence"] for r in dev_sample], [x for x, _ in preds], p)
    U.write_amr_style([r["sentence"] for r in dev_sample], [r["umr"] for r in dev_sample], g)
    return U.score_smatch(p, g), sum(not ok for _, ok in preds)


# ---------------------------------------------------------------- training
def train(model, tok, train_recs, dev_recs, out_dir, epochs=3, lr=2e-4, bs=16, micro_bs=16,
          warmup_frac=0.03, weight_decay=0.0, grad_clip=1.0, dev_subset=500, seed=42,
          device="cuda", log_every=50, grad_ckpt=True):
    """LoRA fine-tuning; after each epoch greedy-decode a fixed dev sample, keep the best adapter.
    Saves `last/` (resume after a Colab disconnect) and `best/` adapters in out_dir."""
    os.makedirs(out_dir, exist_ok=True)
    torch.manual_seed(seed)
    feats, too_long = build_features(train_recs, tok)
    print(f"train examples: {len(feats)}  (dropped as too long: {too_long})")
    dev_sample = random.Random(seed).sample(dev_recs, min(dev_subset, len(dev_recs)))
    if grad_ckpt:
        model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
        model.enable_input_require_grads()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    accum = max(1, bs // micro_bs)
    steps_per_epoch = math.ceil(len(feats) / bs)
    total = steps_per_epoch * epochs
    warm = max(1, int(warmup_frac * total))
    sched = torch.optim.lr_scheduler.LambdaLR(        # linear warmup, cosine decay
        opt, lambda s: (s + 1) / warm if s < warm else
        0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))

    state_path = os.path.join(out_dir, "last", "train_state.pt")
    start_epoch, best = 0, -1.0
    if os.path.exists(state_path):
        from peft import set_peft_model_state_dict
        from safetensors.torch import load_file
        st = torch.load(state_path, map_location="cpu")
        set_peft_model_state_dict(model, load_file(os.path.join(out_dir, "last", "adapter_model.safetensors")))
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"])
        start_epoch, best = st["epoch"] + 1, st["best"]
        print(f"resumed after epoch {start_epoch}, best dev SMATCH so far {best:.2f}")

    for ep in range(start_epoch, epochs):
        model.train()
        t0, run, step = time.time(), 0.0, 0
        for k, b in enumerate(batches(feats, bs, shuffle=True, seed=seed + ep)):
            for m in range(0, len(b), micro_bs):
                inp, att, lab = collate([feats[i] for i in b[m:m + micro_bs]], tok.pad_token_id, device)
                with torch.autocast("cuda", dtype=torch.bfloat16, enabled=(device == "cuda")):
                    logits = model(input_ids=inp, attention_mask=att).logits
                # next-token loss, averaged over graph tokens of the whole batch
                lg, lb = logits[:, :-1].float(), lab[:, 1:]
                n_tok = (lab[:, 1:] != -100).sum()
                loss = F.cross_entropy(lg.reshape(-1, lg.size(-1)), lb.reshape(-1),
                                       ignore_index=-100, reduction="sum") / max(1, n_tok.item())
                (loss * len(b[m:m + micro_bs]) / len(b)).backward()
                run += loss.item() * len(b[m:m + micro_bs]) / len(b)
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(params, grad_clip)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
            step += 1
            if step % log_every == 0:
                print(f"epoch {ep + 1} step {step}/{steps_per_epoch} loss {run / log_every:.4f} "
                      f"lr {sched.get_last_lr()[0]:.2e} ({time.time() - t0:.0f}s)")
                run = 0.0

        sm, invalid = dev_smatch(model, tok, dev_sample, os.path.join(out_dir, "dev_tmp"), device)
        rec = {"epoch": ep + 1, "dev_smatch": round(sm, 2), "dev_invalid": invalid,
               "minutes": round((time.time() - t0) / 60, 1)}
        print(rec)
        with open(os.path.join(out_dir, "log.jsonl"), "a") as f:
            f.write(json.dumps(rec) + "\n")
        if sm > best:
            best = sm
            model.save_pretrained(os.path.join(out_dir, "best"))
            print(f"  new best -> saved {out_dir}/best")
        model.save_pretrained(os.path.join(out_dir, "last"))
        torch.save({"opt": opt.state_dict(), "sched": sched.state_dict(), "epoch": ep, "best": best},
                   state_path)
    return best

Overwriting llm_tools.py


# Part 1 — Data

### 1.1 Load Job 1's splits
Reusing the saved files guarantees the LLM sees **exactly** the same train / dev / test sentences as AMRBART.

In [6]:
import umr_tools as U, llm_tools as L
A = {k: U.load_jsonl(f'{DRIVE}/data/A/{k}.jsonl') for k in ('train', 'dev', 'test')}
B = {k: U.load_jsonl(f'{DRIVE}/data/B/{k}.jsonl') for k in ('train', 'dev', 'test')}
print('A', {k: len(v) for k, v in A.items()}, '| B', {k: len(v) for k, v in B.items()})

A {'train': 21460, 'dev': 5897, 'test': 2941} | B {'train': 24214, 'dev': 1350, 'test': 1358}


### 1.2 The one-line graph format 🔧 (you chose)
Same graph, written on one line, variables renamed `z0, z1, …` in order of appearance. Annotator names like `s34a` encode sentence numbers — pure noise for the model. Renaming never changes the score (checked in 1.3).

In [7]:
x = A['test'][0]
print('sentence :', x['sentence'])
print('gold     :', x['umr'])
print('LLM sees :', L.PROMPT.format(sent=x['sentence']), L.to_oneline(x['umr']))

sentence : [Builder picks up a orange block at X:5 Y:2 Z:-5]
gold     : (s14p / pick-up-04
    :ARG0 (s14b / builder)
    :ARG1 (s14b2 / block
        :color (s14o / orange))
    :ARG2 (s14s / space
        :place (s14c / cartesian-coordinate-entity
            :x 5
            :y 2
            :z -5)))
LLM sees : Parse the sentence into a UMR graph.
Sentence: [Builder picks up a orange block at X:5 Y:2 Z:-5]
UMR: (z0 / pick-up-04 :ARG0 (z1 / builder) :ARG1 (z2 / block :color (z3 / orange)) :ARG2 (z4 / space :place (z5 / cartesian-coordinate-entity :x 5 :y 2 :z -5)))


### 1.3 Sanity check: format → parse back must score 100

In [8]:
t = A['test']
back = [L.from_output(L.to_oneline(x['umr']))[0] for x in t]
U.evaluate_all([x['sentence'] for x in t], back, [x['umr'] for x in t],
               [x['is_minecraft'] for x in t], f'{DRIVE}/eval/llm_format_roundtrip', run_smatchpp=False)['all']

{'n': 2941, 'AnCast': 100.0, 'AnCast_amrfmt': 100.0, 'SMATCH': 100.0}

### 1.4 Load Qwen3-8B-Base + LoRA, check sequence lengths

In [9]:
import torch, numpy as np
model, tok = L.load_model()
lens = [len(tok(L.PROMPT.format(sent=x['sentence']) + ' ' + L.to_oneline(x['umr']))['input_ids']) for x in A['train']]
print('tokens per example: median', int(np.median(lens)), '| 99%', int(np.percentile(lens, 99)), '| max', max(lens))
print('dropped by the 1280-token limit:', sum(l > 1280 for l in lens))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
tokens per example: median 96 | 99% 345 | max 758
dropped by the 1280-token limit: 0


# Part 2 — Train on split A

### 2.1 Train 🔧
| Setting | Value | Why |
|---|---|---|
| LoRA rank / alpha / dropout | 16 / 32 / 0.05 | common default; enough capacity for a fixed output format |
| LoRA targets | all attention + MLP projections | better than attention-only for learning a new output language |
| learning rate | 2e-4 | standard for LoRA (full fine-tuning lr like Job 1's 4e-5 is too small for LoRA) |
| epochs | 3 | LoRA on ~21k examples converges in 2–3 epochs; best epoch kept |
| batch | 16 | same as Job 1 |
| schedule | 3% warmup, cosine decay | standard for LLM fine-tuning |
| loss | graph tokens only (prompt masked) | the model should learn to write graphs, not repeat the prompt |
| precision / memory | bf16 + gradient checkpointing | fits 8B comfortably in 80GB |
| model selection | greedy SMATCH on a fixed 500-sentence dev sample after each epoch | same idea as Job 1 (1,000 there; LLM decoding is slower) |

Checkpoints go to Drive. **If Colab disconnects: re-run Part 0, 1.1, 1.4 and this cell — it resumes after the last finished epoch.**

In [10]:
RUN_A = f'{DRIVE}/runs/qwen3_A'
best = L.train(model, tok, A['train'], A['dev'], RUN_A, epochs=3, lr=2e-4, bs=16)
print('best dev SMATCH (greedy, 500 subset):', best)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


train examples: 21460  (dropped as too long: 0)
epoch 1 step 50/1342 loss 1.0390 lr 8.50e-05 (45s)
epoch 1 step 100/1342 loss 0.2964 lr 1.68e-04 (89s)
epoch 1 step 150/1342 loss 0.1784 lr 2.00e-04 (124s)
epoch 1 step 200/1342 loss 0.1600 lr 2.00e-04 (165s)
epoch 1 step 250/1342 loss 0.1353 lr 1.99e-04 (205s)
epoch 1 step 300/1342 loss 0.1413 lr 1.99e-04 (242s)
epoch 1 step 350/1342 loss 0.1270 lr 1.98e-04 (285s)
epoch 1 step 400/1342 loss 0.1048 lr 1.97e-04 (321s)
epoch 1 step 450/1342 loss 0.1046 lr 1.96e-04 (357s)
epoch 1 step 500/1342 loss 0.1083 lr 1.95e-04 (398s)
epoch 1 step 550/1342 loss 0.0979 lr 1.94e-04 (437s)
epoch 1 step 600/1342 loss 0.0883 lr 1.93e-04 (477s)
epoch 1 step 650/1342 loss 0.0804 lr 1.91e-04 (518s)
epoch 1 step 700/1342 loss 0.0830 lr 1.89e-04 (560s)
epoch 1 step 750/1342 loss 0.0846 lr 1.87e-04 (598s)
epoch 1 step 800/1342 loss 0.0631 lr 1.85e-04 (635s)
epoch 1 step 850/1342 loss 0.0808 lr 1.83e-04 (677s)
epoch 1 step 900/1342 loss 0.0871 lr 1.81e-04 (714s)
e

{'epoch': 1, 'dev_smatch': 87.09, 'dev_invalid': 0, 'minutes': 21.6}
  new best -> saved /content/drive/MyDrive/umr_job1/runs/qwen3_A/best
epoch 2 step 50/1342 loss 0.0481 lr 1.52e-04 (39s)
epoch 2 step 100/1342 loss 0.0501 lr 1.49e-04 (76s)
epoch 2 step 150/1342 loss 0.0584 lr 1.45e-04 (114s)
epoch 2 step 200/1342 loss 0.0431 lr 1.41e-04 (150s)
epoch 2 step 250/1342 loss 0.0436 lr 1.38e-04 (188s)
epoch 2 step 300/1342 loss 0.0692 lr 1.34e-04 (230s)
epoch 2 step 350/1342 loss 0.0485 lr 1.30e-04 (269s)
epoch 2 step 400/1342 loss 0.0604 lr 1.26e-04 (308s)
epoch 2 step 450/1342 loss 0.0517 lr 1.22e-04 (344s)
epoch 2 step 500/1342 loss 0.0473 lr 1.18e-04 (383s)
epoch 2 step 550/1342 loss 0.0366 lr 1.15e-04 (419s)
epoch 2 step 600/1342 loss 0.0376 lr 1.11e-04 (457s)
epoch 2 step 650/1342 loss 0.0334 lr 1.07e-04 (497s)
epoch 2 step 700/1342 loss 0.0434 lr 1.02e-04 (533s)
epoch 2 step 750/1342 loss 0.0459 lr 9.85e-05 (578s)
epoch 2 step 800/1342 loss 0.0383 lr 9.45e-05 (616s)
epoch 2 step 850

### 2.2 Test with the best adapter 🔧
**Greedy decoding** (no beam): standard for LLM parsers and ~5× faster than beam 5. Note AMRBART in Job 1 used beam 5 — mention this difference in the write-up.

In [11]:
import json, gc, pandas as pd
from peft import PeftModel
def load_best(run):
    """Free the GPU, then load the base model + the best LoRA adapter of `run`."""
    global model
    del model; gc.collect(); torch.cuda.empty_cache()
    m, _ = L.load_model(adapter=f'{run}/best')
    return m
model = load_best(RUN_A)
t = A['test']
preds_A = L.predict(model, tok, [x['sentence'] for x in t])
U.save_jsonl([{**x, 'pred': g, 'valid': ok} for x, (g, ok) in zip(t, preds_A)], f'{RUN_A}/test_predictions.jsonl')
res_A = U.evaluate_all([x['sentence'] for x in t], [g for g, _ in preds_A], [x['umr'] for x in t],
                       [x['is_minecraft'] for x in t], f'{RUN_A}/eval')
amrbart_A = json.load(open(f'{DRIVE}/runs/amrbart_A/eval/test_scores.json'))
print('invalid predictions:', res_A['invalid_predictions'])
cols = ['AnCast', 'SMATCH', 'SMATCH++']
pd.DataFrame({'Paper AMRBART': [81.70, 86.24, 88.70], 'Paper BiBL (best)': [84.35, 88.82, 90.98],
              'AMRBART (Job 1)': [amrbart_A['all'][c] for c in cols],
              'Qwen3-8B LoRA': [res_A['all'][c] for c in cols]}, index=cols)

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
invalid predictions: 0


,Paper AMRBART,Paper BiBL (best),AMRBART (Job 1),Qwen3-8B LoRA
AnCast,81.70,84.35,86.85,84.01
SMATCH,86.24,88.82,91.14,89.10
SMATCH++,88.70,90.98,91.15,89.12


### 2.3 Minecraft vs non-Minecraft (the weakness from Job 1)

In [12]:
pd.DataFrame({f'{m} – {s}': {c: r[s][c] for c in cols}
              for m, r in (('AMRBART', amrbart_A), ('Qwen3', res_A)) for s in ('minecraft', 'non_minecraft')}).T

,AnCast,SMATCH,SMATCH++
AMRBART – minecraft,92.64,94.84,94.85
AMRBART – non_minecraft,66.82,77.97,77.95
Qwen3 – minecraft,91.87,94.30,94.30
Qwen3 – non_minecraft,57.01,70.69,70.79


# Part 3 — Train on clean split B

Fresh LoRA adapter, identical recipe (split-B test sentences are in split-A train).

In [13]:
del model; gc.collect(); torch.cuda.empty_cache()
model, tok = L.load_model()
RUN_B = f'{DRIVE}/runs/qwen3_B'
L.train(model, tok, B['train'], B['dev'], RUN_B, epochs=3, lr=2e-4, bs=16)
model = load_best(RUN_B)
t = B['test']
preds_B = L.predict(model, tok, [x['sentence'] for x in t])
U.save_jsonl([{**x, 'pred': g, 'valid': ok} for x, (g, ok) in zip(t, preds_B)], f'{RUN_B}/test_predictions.jsonl')
res_B = U.evaluate_all([x['sentence'] for x in t], [g for g, _ in preds_B], [x['umr'] for x in t],
                       [x['is_minecraft'] for x in t], f'{RUN_B}/eval')
print('invalid predictions:', res_B['invalid_predictions'])
pd.DataFrame({k: res_B[k] for k in ('all', 'minecraft', 'non_minecraft')}).T

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
train examples: 24214  (dropped as too long: 0)
epoch 1 step 50/1514 loss 1.0078 lr 7.50e-05 (41s)
epoch 1 step 100/1514 loss 0.4020 lr 1.49e-04 (85s)
epoch 1 step 150/1514 loss 0.1937 lr 2.00e-04 (131s)
epoch 1 step 200/1514 loss 0.1904 lr 2.00e-04 (175s)
epoch 1 step 250/1514 loss 0.1350 lr 2.00e-04 (213s)
epoch 1 step 300/1514 loss 0.0915 lr 1.99e-04 (249s)
epoch 1 step 350/1514 loss 0.1197 lr 1.99e-04 (287s)
epoch 1 step 400/1514 loss 0.1118 lr 1.98e-04 (328s)
epoch 1 step 450/1514 loss 0.1077 lr 1.98e-04 (370s)
epoch 1 step 500/1514 loss 0.0956 lr 1.97e-04 (406s)
epoch 1 step 550/1514 loss 0.0995 lr 1.96e-04 (444s)
epoch 1 step 600/1514 loss 0.0929 lr 1.95e-04 (481s)
epoch 1 step 650/1514 loss 0.0550 lr 1.93e-04 (516s)
epoch 1 step 700/1514 loss 0.0876 lr 1.92e-04 (556s)
epoch 1 step 750/1514 loss 0.0987 lr 1.91e-04 (596s)
epoch 1 step 800/1514 loss 0.0818 lr 1.89e-04 (633s)
epoch 1 step 850/1514 loss 

Error processing (z0 / say-01 :ARG0 (z1 / person :name (z2 / name :op1 "Cunliffe") :wiki "Michael_Cunliffe") :ARG1 (z3 / decide-01 :ARG1 (z4 / or :op1 (z5 / collect-01 :ARG1 (z6 / information :mod (z7 / biometric)) :ARG2 (z8 / body :possessor (z9 / person))) :op2 (z10 / collect-01 :ARG1 z6 :manner) manner


{'epoch': 1, 'dev_smatch': 79.9, 'dev_invalid': 2, 'minutes': 26.2}
  new best -> saved /content/drive/MyDrive/umr_job1/runs/qwen3_B/best
epoch 2 step 50/1514 loss 0.0449 lr 1.52e-04 (41s)
epoch 2 step 100/1514 loss 0.0497 lr 1.49e-04 (80s)
epoch 2 step 150/1514 loss 0.0468 lr 1.46e-04 (114s)
epoch 2 step 200/1514 loss 0.0541 lr 1.43e-04 (152s)
epoch 2 step 250/1514 loss 0.0527 lr 1.40e-04 (191s)
epoch 2 step 300/1514 loss 0.0460 lr 1.37e-04 (229s)
epoch 2 step 350/1514 loss 0.0482 lr 1.33e-04 (264s)
epoch 2 step 400/1514 loss 0.0589 lr 1.30e-04 (306s)
epoch 2 step 450/1514 loss 0.0542 lr 1.26e-04 (344s)
epoch 2 step 500/1514 loss 0.0531 lr 1.23e-04 (384s)
epoch 2 step 550/1514 loss 0.0590 lr 1.19e-04 (428s)
epoch 2 step 600/1514 loss 0.0457 lr 1.16e-04 (468s)
epoch 2 step 650/1514 loss 0.0441 lr 1.12e-04 (506s)
epoch 2 step 700/1514 loss 0.0322 lr 1.09e-04 (545s)
epoch 2 step 750/1514 loss 0.0521 lr 1.05e-04 (586s)
epoch 2 step 800/1514 loss 0.0493 lr 1.02e-04 (629s)
epoch 2 step 850/

Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

trainable params: 43,646,976 || all params: 8,234,382,336 || trainable%: 0.5301
invalid predictions: 0


,n,AnCast,AnCast_amrfmt,SMATCH,SMATCH++
all,1358.0,75.54,76.11,83.53,83.55
minecraft,1001.0,82.53,82.81,88.34,88.34
non_minecraft,357.0,59.71,60.87,72.39,72.44


# Part 4 — Compare with AMRBART

### 4.1 Final table (goes in the write-up)

In [14]:
amrbart_B = json.load(open(f'{DRIVE}/runs/amrbart_B/eval/test_scores.json'))
rows = {'Paper AMRBART (Table 5)': {'AnCast': 81.70, 'SMATCH': 86.24, 'SMATCH++': 88.70},
        'Paper BiBL (Table 5)': {'AnCast': 84.35, 'SMATCH': 88.82, 'SMATCH++': 90.98}}
for split, ra, rq in (('A', amrbart_A, res_A), ('B', amrbart_B, res_B)):
    for sub in ('all', 'minecraft', 'non_minecraft'):
        rows[f'AMRBART {split} – {sub}'] = {k: ra[sub][k] for k in ('n', 'AnCast', 'SMATCH', 'SMATCH++')}
        rows[f'Qwen3   {split} – {sub}'] = {k: rq[sub][k] for k in ('n', 'AnCast', 'SMATCH', 'SMATCH++')}
final = pd.DataFrame(rows).T
final.to_csv(f'{DRIVE}/job2_results.csv')
print(final.round(2).to_string())

                           AnCast  SMATCH  SMATCH++       n
Paper AMRBART (Table 5)     81.70   86.24     88.70     NaN
Paper BiBL (Table 5)        84.35   88.82     90.98     NaN
AMRBART A – all             86.85   91.14     91.15  2941.0
Qwen3   A – all             84.01   89.10     89.12  2941.0
AMRBART A – minecraft       92.64   94.84     94.85  2478.0
Qwen3   A – minecraft       91.87   94.30     94.30  2478.0
AMRBART A – non_minecraft   66.82   77.97     77.95   463.0
Qwen3   A – non_minecraft   57.01   70.69     70.79   463.0
AMRBART B – all             79.35   86.19     86.18  1358.0
Qwen3   B – all             75.54   83.53     83.55  1358.0
AMRBART B – minecraft       84.23   89.26     89.27  1001.0
Qwen3   B – minecraft       82.53   88.34     88.34  1001.0
AMRBART B – non_minecraft   68.42   79.10     79.09   357.0
Qwen3   B – non_minecraft   59.71   72.39     72.44   357.0


### 4.2 Sentence by sentence on non-Minecraft (split A) 🔧
Per-sentence SMATCH for AMRBART vs Qwen3 on the **same** test sentences: how often each wins, plus a few examples of each. This is your error analysis for the write-up (the paper's §4.2 does the same with two hand-picked examples).

In [15]:
import os
def sent_smatch(pred, gold):
    p, g = '/tmp/p1.txt', '/tmp/g1.txt'
    U.write_amr_style(['x'], [U.to_valid(pred)[0]], p); U.write_amr_style(['x'], [gold], g)
    return U.score_smatch(p, g)
ab = {x['sentence'] + x['umr']: x['pred'] for x in U.load_jsonl(f'{DRIVE}/runs/amrbart_A/test_predictions.jsonl')}
qw = U.load_jsonl(f'{RUN_A}/test_predictions.jsonl')
rows = []
for x in qw:
    if x['is_minecraft']: continue
    a = sent_smatch(ab[x['sentence'] + x['umr']], x['umr']); q = sent_smatch(x['pred'], x['umr'])
    rows.append({'sentence': x['sentence'], 'amrbart': a, 'qwen3': q, 'diff': q - a,
                 'gold': x['umr'], 'pred_amrbart': ab[x['sentence'] + x['umr']], 'pred_qwen3': x['pred']})
cmp = pd.DataFrame(rows)
print(f"non-Minecraft sentences: {len(cmp)} | Qwen3 better: {(cmp['diff'] > 1).sum()} | "
      f"AMRBART better: {(cmp['diff'] < -1).sum()} | tie: {(cmp['diff'].abs() <= 1).sum()}")
cmp.to_csv(f'{DRIVE}/job2_nonminecraft_comparison.csv', index=False)
for title, sub in (('Qwen3 much better', cmp.nlargest(3, 'diff')), ('AMRBART much better', cmp.nsmallest(3, 'diff'))):
    print('\n=====', title)
    for _, r in sub.iterrows():
        print(f"\n{r['sentence']}\n  AMRBART {r['amrbart']:.0f} | Qwen3 {r['qwen3']:.0f}\n--gold\n{r['gold']}\n--AMRBART\n{r['pred_amrbart']}\n--Qwen3\n{r['pred_qwen3']}")

non-Minecraft sentences: 463 | Qwen3 better: 86 | AMRBART better: 238 | tie: 139

===== Qwen3 much better

In other words, Congress should stop all the politcal bickering, put their heads together and start working things out in a real bi-partisan format.
  AMRBART 4 | Qwen3 79
--gold
(s4r / recommend-01
    :ARG1 (s4a / and
        :op1 (s4s / stop-01
            :ARG0 (s4g2 / government-organization
                :part (s4h / head)
                :name (s4n / name
                    :op1 "Congress")
                :wiki "United_States_Congress")
            :ARG1 (s4b / bicker-01
                :ARG0 s4g2
                :ARG1 (s4p / politics)
                :mod (s4a2 / all)))
        :op2 (s4p2 / put-01
            :ARG0 s4g2
            :ARG1 s4h
            :ARG2 (s4t / together))
        :op3 (s4s2 / start-01
            :ARG0 s4g2
            :ARG1 (s4w / work-out-02
                :ARG0 s4g2
                :ARG1 (s4t2 / thing)
                :manner (s4b2 / bipartisa

### What to take from Job 2
1. Does a modern LLM beat the paper's best parser (BiBL) and your AMRBART — overall and on **non-Minecraft**?
2. Does the gap hold on the **clean split B** (not just thanks to duplicates)?
3. Concrete examples where each model wins (4.2).
Write-up caveats: one run per setting; LLM uses greedy decoding, AMRBART beam 5; test set is our random sample (the paper's is unreleased).